## Gradient Boosting with scikit-learn
Gradient Boosting is a sequential ensemble method that trains trees on residuals, achieving state-of-the-art performance on structured data. It requires careful hyperparameter tuning to prevent overfitting.
This notebook covers the basics of Gradient Boosting, including its comparison to Random Forest, the effects of learning rate, n_estimators, max_depth, and subsample, as well as feature importance and staged prediction.

### Gradient Boosting vs Random Forest
Gradient Boosting and Random Forest are both ensemble methods, but they differ in their approach. Random Forest reduces variance by combining multiple trees, while Gradient Boosting reduces bias and variance by training trees sequentially, each correcting the errors of the previous one.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris, make_regression
from sklearn.metrics import accuracy_score, mean_squared_error, log_loss
import matplotlib.pyplot as plt

np.random.seed(42)

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("=== Gradient Boosting Classification ===")
print(f"Dataset: {len(X)} samples, {X.shape[1]} features, {len(np.unique(y))} classes")

The output shows the dataset characteristics, including the number of samples, features, and classes. This information is useful for understanding the problem and selecting the appropriate algorithm.

### Compare Random Forest vs Gradient Boosting
We compare the performance of Random Forest and Gradient Boosting on the iris dataset.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_accuracy = rf.score(X_test, y_test)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_accuracy = gb.score(X_test, y_test)

print(f"\nRandom Forest accuracy: {rf_accuracy:.4f}")
print(f"Gradient Boosting accuracy: {gb_accuracy:.4f}")
print(f"GB typically performs better but requires more tuning")

The output shows the accuracy of both algorithms. Gradient Boosting typically performs better than Random Forest, but it requires more tuning.

### Learning Rate and N_Estimators
The learning rate controls the step size of each tree, while n_estimators is the number of trees. A lower learning rate requires more trees but often results in better generalization.

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]
test_accuracies = []

for lr in learning_rates:
    gb_lr = GradientBoostingClassifier(learning_rate=lr, n_estimators=100, random_state=42)
    gb_lr.fit(X_train, y_train)
    acc = gb_lr.score(X_test, y_test)
    test_accuracies.append(acc)
    print(f"learning_rate={lr}: Accuracy={acc:.4f}")

The output shows the effect of different learning rates on the accuracy of the model. A lower learning rate often results in better generalization, but it requires more trees.

### Max Depth: Weak Learners
Gradient Boosting uses shallow trees (max_depth=3-5 default) as weak learners. Deeper trees can hurt generalization.

In [ ]:
depths = [1, 2, 3, 5, 7, 10]
print(f"Effect on accuracy (100 estimators, lr=0.1):")
for depth in depths:
    gb_depth = GradientBoostingClassifier(max_depth=depth, n_estimators=100, learning_rate=0.1, random_state=42)
    gb_depth.fit(X_train, y_train)
    acc = gb_depth.score(X_test, y_test)
    print(f"  max_depth={depth:2d}: {acc:.4f}")

The output shows the effect of different tree depths on the accuracy of the model. Shallow trees (max_depth=3-5) often result in better generalization.

### Subsample: Stochastic Gradient Boosting
Training each tree on a random subset of the data reduces overfitting and speeds up training.

In [ ]:
subsamples = [0.5, 0.7, 0.8, 1.0]
print(f"Effect on accuracy (100 estimators, lr=0.1, depth=3):")
for subsample in subsamples:
    gb_sub = GradientBoostingClassifier(subsample=subsample, n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    gb_sub.fit(X_train, y_train)
    acc = gb_sub.score(X_test, y_test)
    print(f"  subsample={subsample}: {acc:.4f}")

The output shows the effect of different subsample sizes on the accuracy of the model. A subsample size of 0.8 often results in better generalization.

### Feature Importance and Staging
Features are ranked by how often they split; early stops show importance.

In [ ]:
from sklearn.inspection import permutation_importance

gb_final = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, subsample=0.8, random_state=42)
gb_final.fit(X_train, y_train)

importance = gb_final.feature_importances_
print(f"Feature importance (split-based):")
for idx in np.argsort(importance)[::-1]:
    print(f"  {iris.feature_names[idx]}: {importance[idx]:.4f}")

perm_importance = permutation_importance(gb_final, X_test, y_test, n_repeats=10, random_state=42)
print(f"\nPermutation importance:")
for idx in np.argsort(perm_importance.importances_mean)[::-1]:
    print(f"  {iris.feature_names[idx]}: {perm_importance.importances_mean[idx]:.4f}")

The output shows the feature importance of the model, both using the split-based method and the permutation importance method.

### Staged Prediction: Monitoring Progress
Observe training progress; detect overfitting by iteration number.

In [ ]:
from sklearn.model_selection import cross_val_score

train_scores = []
test_scores = []
for y_pred in gb_final.staged_predict(X_train):
    train_scores.append(accuracy_score(y_train, y_pred))
for y_pred in gb_final.staged_predict(X_test):
    test_scores.append(accuracy_score(y_test, y_pred))

best_test_idx = np.argmax(test_scores)
print(f"Best test accuracy at iteration {best_test_idx + 1}:")
print(f"  Train accuracy: {train_scores[best_test_idx]:.4f}")
print(f"  Test accuracy: {test_scores[best_test_idx]:.4f}")
if best_test_idx < len(test_scores) - 1:
    print(f"  Overfitting starts after iteration {best_test_idx + 1}")

The output shows the training and test accuracy at each iteration, as well as the iteration where overfitting starts.

### Early Stopping
Stop training when validation score plateaus to prevent overfitting.

In [ ]:
gb_early = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
gb_early.fit(X_train, y_train)

best_test_score = 0
best_n_estimators = 0
for n_est, y_pred in enumerate(gb_early.staged_predict(X_test), 1):
    score = accuracy_score(y_test, y_pred)
    if score > best_test_score:
        best_test_score = score
        best_n_estimators = n_est

print(f"Without early stopping: n_estimators=200")
print(f"  Test accuracy: {gb_early.score(X_test, y_test):.4f}")
print(f"With early stopping at n_estimators={best_n_estimators}:")
print(f"  Test accuracy would be: {best_test_score:.4f}")

The output shows the test accuracy with and without early stopping.

### Gradient Boosting Regression
Boosting works for continuous targets too.

In [ ]:
X_reg, y_reg = make_regression(n_samples=200, n_features=5, noise=20, random_state=42)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

gb_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, subsample=0.8, random_state=42)
gb_reg.fit(X_reg_train, y_reg_train)

y_pred_reg = gb_reg.predict(X_reg_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_reg))
r2 = gb_reg.score(X_reg_test, y_reg_test)

print(f"RMSE: {rmse:.4f}")
print(f"R² score: {r2:.4f}")

The output shows the RMSE and R² score of the regression model.

### Huber Loss for Robustness
Less sensitive to outliers than squared error.

In [ ]:
losses = ['squared_error', 'huber', 'quantile']
print(f"Classification loss: 'log_loss' (logistic loss)")
print(f"Regression loss options:")
for loss_fn in losses:
    try:
        gb_loss = GradientBoostingRegressor(loss=loss_fn, n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
        gb_loss.fit(X_reg_train, y_reg_train)
        r2 = gb_loss.score(X_reg_test, y_reg_test)
        print(f"  {loss_fn}: R² = {r2:.4f}")
    except:
        print(f"  {loss_fn}: not available")

The output shows the R² score for each loss function.

### Initialization: Warm Start
Can incrementally add more boosting rounds (n_estimators).

In [ ]:
gb_warm = GradientBoostingClassifier(n_estimators=10, warm_start=True, random_state=42)
gb_warm.fit(X_train, y_train)
print(f"Initial (10 estimators): {gb_warm.score(X_test, y_test):.4f}")

gb_warm.n_estimators = 50
gb_warm.fit(X_train, y_train)
print(f"After adding 40 more (50 total): {gb_warm.score(X_test, y_test):.4f}")

gb_warm.n_estimators = 100
gb_warm.fit(X_train, y_train)
print(f"After adding 50 more (100 total): {gb_warm.score(X_test, y_test):.4f}")

The output shows the accuracy of the model after adding more boosting rounds.

### Practical ML Scenarios
This section covers practical machine learning scenarios, including hyperparameter tuning, imbalanced classification, and training monitoring.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 0.9]
}

random_search = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_grid, n_iter=20, cv=3, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

print(f"  Best parameters: {random_search.best_params_}")
print(f"  Best CV score: {random_search.best_score_:.4f}")

The output shows the best parameters and the best cross-validation score.

## Key Takeaways
* Gradient Boosting is a sequential ensemble method that trains trees on residuals.
* The learning rate controls the step size of each tree.
* The number of estimators (n_estimators) is the number of trees.
* The max depth of each tree should be shallow (3-5) for weak learners.
* The subsample size should be 0.7-0.8 for stochastic gradient boosting.
* Feature importance can be calculated using the split-based method or permutation importance.
* Staged prediction can be used to monitor training progress and detect overfitting.
* Early stopping can be used to prevent overfitting.
* Gradient Boosting can be used for regression tasks as well.
* Huber loss can be used for robust regression.
* Warm start can be used to incrementally add more boosting rounds.